# ML Training Notebook
DummyClassifier
RandomForestClassifier
LogisticRegression
MLflow tracking
Learning curves

data_storage_gcs.ipynb
    ↓
train.parquet
evaluation.parquet
test.parquet
    ↓
feature_engineering.ipynb
    ↓
train_features.parquet
evaluation_features.parquet
test_features.parquet
    ↓
model_training_mlflow.ipynb

train_df = pd.read_parquet(
    PROJECT_ROOT /
    "data" /
    "processed" /
    "train.parquet"
)

eval_df = pd.read_parquet(
    PROJECT_ROOT /
    "data" /
    "processed" /
    "evaluation.parquet"
)

test_df = pd.read_parquet(
    PROJECT_ROOT /
    "data" /
    "processed" /
    "test.parquet"
)

#assert set(df["Quality"].unique()).issubset({0, 1, 3})

In [4]:
from pathlib import Path
import os

from dotenv import load_dotenv
from src.config import (
    PROJECT_ID,
    BUCKET_NAME,
    LOCATION
)

import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


from sklearn.model_selection import learning_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

import warnings
import time
from google.cloud import storage

warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'src'

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(NN_RANDOM_STATE)
np.random.seed(NN_RANDOM_STATE)

## Model Hyperparameter Configuration

In [1]:
# Define hyperparameters for SVM and Deep Neural Network
SVM_RANDOM_STATE = 42
SVM_MAX_ITER = 1000

# PyTorch Neural Network Hyperparameters
NN_INPUT_SIZE = 8  # Number of preprocessed features (Pclass, Age, SibSp, Parch, Fare, Sex_male, Embarked_Q, Embarked_S)
NN_HIDDEN_SIZE = 1000
NN_EPOCHS = 100
NN_BATCH_SIZE = 64
NN_LEARNING_RATE = 0.001
NN_RANDOM_STATE = 42

# Device selection
DEVICE = 'cpu' #normally it would be torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")


NameError: name 'warnings' is not defined

## Load Engineered Dataset

In [ ]:
PROJECT_ROOT = Path.cwd().parent

from src.config import (
    PROJECT_ID,
    BUCKET_NAME,
    LOCATION,
    PROCESSED_DIR,
    FEATURE_DIR
)

df = pd.read_parquet(
    PROJECT_ROOT /
    "data" /
    "features" /
    "air_quality_features.parquet"
)

print(df.shape)

df.head()

## Prepare Features and Target

In [ ]:
exclude_cols = [
    "Datetime",
    "Target_Next_Hour",
    "Date",
    "Time"
]

train_df = create_features(train_df)

eval_df = create_features(eval_df)

test_df = create_features(test_df)

In [ ]:
# Candidate predictors: current measurements only
predictors = measurement_cols.copy()

# Prepare modelling dataset
model_df = forecast_df[
    ["Datetime"] + predictors + ["Target_Next_Hour"]
].dropna(subset=["Target_Next_Hour"])

# Chronological 80/20 train-test split
split_index = int(len(model_df) * 0.8)

train = model_df.iloc[:split_index]
test = model_df.iloc[split_index:]

X_train = train[predictors]
X_test = test[predictors]

y_train = train["Target_Next_Hour"]
y_test = test["Target_Next_Hour"]

# Remove predictors completely missing in the training data
usable_predictors = X_train.columns[
    X_train.notna().any()
].tolist()

X_train = X_train[usable_predictors]
X_test = X_test[usable_predictors]

# Baseline: always predict the most frequent training category
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

# Random Forest pipeline
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_predictions = rf_pipeline.predict(X_test)

# Compare model performance
results = pd.DataFrame({
    "model": ["Baseline", "Random Forest"],
    "accuracy": [
        baseline.score(X_test, y_test),
        rf_pipeline.score(X_test, y_test)
    ],
    "balanced_accuracy": [
        balanced_accuracy_score(y_test, baseline_predictions),
        balanced_accuracy_score(y_test, rf_predictions)
    ],
    "macro_f1": [
        f1_score(y_test, baseline_predictions, average="macro"),
        f1_score(y_test, rf_predictions, average="macro")
    ]
})

display(results.round(3))

# Detailed Random Forest results
print("Random Forest classification report:")
print(classification_report(
    y_test,
    rf_predictions,
    zero_division=0
))